In [ ]:
#
# Import Visualization FrameWork Tools
import pandas as pd
import numpy as np

# import missingno as msno

In [ ]:
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/new_notebooks/nyc-airbnb/AB_NYC_2019.csv")
factor = 30
df = pd.concat([df] * factor)
df.head()

In [ ]:
### cell 1 ###

df.shape

In [ ]:
### cell 2 ###

# check null-value and each columns' dtype to do EDA.
print(df.info())

In [ ]:
### cell 3 ###

df = df.drop(["id", "last_review", "name", "host_name"], axis=1)
df.head()

In [ ]:
### cell 4 ###

# Filter rows where availability_365 ≠ 0 and reset the index in one go
df = df[df["availability_365"] != 0].reset_index()
df.head()

In [ ]:
### cell 5 ###

df["reviews_per_month"] = df["reviews_per_month"].fillna(0)
df.head()

In [ ]:
### cell 6 ###

df.isnull().sum()

In [ ]:
### cell 7 ###

df.shape

In [ ]:
### cell 8 ###

nyc_sub_1 = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]
# Use a single groupby to collect prices into lists, then reindex to preserve original order
y_1 = df.groupby("neighbourhood_group")["price"].agg(list).reindex(nyc_sub_1).tolist()
y_1

In [ ]:
### cell 9 ###

# (Re-declare the ordering list if it isn’t already in scope)
nyc_sub_2 = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]

# Vectorized, chained version equivalent to the original one-liner
y_optimized = (
    df.groupby("neighbourhood_group", sort=False)["price"]
    .agg(list)
    .reindex(nyc_sub_2)
    .tolist()
)

In [ ]:
### cell 10 ###

df["price"].values[0]

In [ ]:
### cell 11 ###

# Separate by each price
df["price_group"] = 0
# Get the first 100 row labels
tmp_idx = df.index[:100]
# Define bins matching the original thresholds
bins = [-float("inf"), 50, 250, 500, 750, 1000, float("inf")]
# Vectorized categorization for the first 100 rows
df.loc[tmp_idx, "price_group"] = pd.cut(
    df.loc[tmp_idx, "price"], bins=bins, right=False
).cat.codes

In [ ]:
### cell 12 ###

suburb_area = [59100000, 183400000, 464900000, 109000000, 151500000]
# Precompute counts once
group_counts = df["neighbourhood_group"].value_counts().reindex(nyc_sub_2)
# Vectorized computation
distance_room_to_room = (
    pd.Series(suburb_area, index=nyc_sub_2) / group_counts / 1000
).tolist()
distance_room_to_room

In [ ]:
### cell 13 ###

grouped = df.groupby("neighbourhood_group")

# Total counts and average price per group
total_counts = grouped.size()
price_total_mean = grouped["price"].mean().round(3)

# Filter once for prices between 50 and 250
mask = (df["price"] > 50) & (df["price"] < 250)
df_filtered = df.loc[mask]

# Counts and average price for filtered data
grouped_f = df_filtered.groupby("neighbourhood_group")
counts_50_250 = grouped_f.size()
price_50_250_mean = grouped_f["price"].mean().round(3)

# Reindex to preserve order of nyc_sub_2
total_amount_sub = total_counts.reindex(nyc_sub_2, fill_value=0).tolist()
total_amount_sub_50_250 = counts_50_250.reindex(nyc_sub_2, fill_value=0).tolist()
aver_price_total = price_total_mean.reindex(nyc_sub_2).tolist()
aver_price_50_250 = price_50_250_mean.reindex(nyc_sub_2).tolist()

# Results
total_amount_sub, total_amount_sub_50_250, aver_price_total, aver_price_50_250

In [ ]:
### cell 14 ###

# Efficient computation using vectorized operations
# 1. Get neighbourhoods ordered by frequency
counts = df["neighbourhood"].value_counts()
# 2. Compute mean price and representative group per neighbourhood, then reindex to the frequency order
grp = (
    df.groupby("neighbourhood")
    .agg(
        price_mean=("price", "mean"),
        neighbourhood_group=("neighbourhood_group", "first"),
    )
    .reindex(counts.index)
)
# 3. Extract the results into the desired lists
nei_x_1 = grp.index.tolist()
nei_y_1 = grp["price_mean"].tolist()
# 4. Map neighbourhood_group to colors
_color_map = {
    "Manhattan": "red",
    "Brooklyn": "blue",
    "Queens": "green",
    "Bronx": "orange",
    "Staten Island": "purple",
}
colors_1 = grp["neighbourhood_group"].map(_color_map).tolist()
# The variables nei_x_1, nei_y_1, colors_1 now match the original outputs

In [ ]:
### cell 15 ###

# Compute the threshold once
thr = round(np.mean(nei_y_1), 3)

# Compute each neighbourhood’s average price and preserve the original iteration order
group_mean = (
    df.groupby("neighbourhood")["price"]
    .mean()
    .reindex(df["neighbourhood"].value_counts().index)
)

# Split into high and low lists in one pass
mask = group_mean >= thr
nei_x_aver_h_1 = group_mean.index[mask].tolist()
nei_y_aver_h_1 = group_mean[mask].tolist()
nei_x_aver_l_1 = group_mean.index[~mask].tolist()
nei_y_aver_l_1 = group_mean[~mask].tolist()

In [ ]:
### cell 16 ###

# Precompute a mapping from neighbourhood to its neighbourhood_group
neigh_group = df.drop_duplicates("neighbourhood").set_index("neighbourhood")[
    "neighbourhood_group"
]
# Define the colour map
echo_map = {
    "Manhattan": "red",
    "Brooklyn": "blue",
    "Queens": "green",
    "Bronx": "orange",
    "Staten Island": "purple",
}
# Build colours_2 by looking up each neighbourhood in two cheap dict/Series lookups
colors_2 = [echo_map[neigh_group[n]] for n in nei_x_aver_h_1]

In [ ]:
### cell 17 ###

# Build a mapping from neighbourhood to its group
neigh_to_group = df.drop_duplicates("neighbourhood").set_index("neighbourhood")[
    "neighbourhood_group"
]

# Define colour for each group
group_to_colour = {
    "Manhattan": "red",
    "Brooklyn": "blue",
    "Queens": "green",
    "Bronx": "orange",
    "Staten Island": "purple",
}

# Vectorise the lookup and colour assignment
colors_3 = pd.Series(nei_x_aver_h_1).map(neigh_to_group).map(group_to_colour).tolist()

In [ ]:
### cell 18 ###

# Compute value_counts and sort once, then extract index and values
tmp = df["calculated_host_listings_count"].value_counts().sort_index()
x_2 = tmp.index.astype(str)
y_2 = tmp.values

x_2, y_2